# Day 48: Talk to Your AI — Out Loud

Every demo so far has been *typing*. Today we give our AI a **voice** and **ears**: you speak a question, it *hears* you, thinks, and *speaks* the answer back.

A study buddy you can actually talk to — built on the free Gemini API, no extra libraries.

## Install

In [2]:
%pip install google-genai numpy --quiet


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Setup

Two models do the work: `gemini-2.5-flash-preview-tts` is the **voice** (text → speech) and `gemini-2.5-flash` is the **ears + brain** (audio in → answer). Three tiny helpers glue them together.

In [15]:
import os, io, wave
import numpy as np
from google import genai
from google.genai import types
from IPython.display import Audio
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')
client = genai.Client(api_key=os.environ["GEMINI_API_KEY2"])

def tts(text, voice="Kore"):
    """AI speaks: returns raw PCM audio (24kHz, 16-bit, mono)."""
    r = client.models.generate_content(
        model="gemini-2.5-flash-preview-tts",
        contents=text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=voice)))))
    return r.candidates[0].content.parts[0].inline_data.data

def to_wav(pcm, rate=24000):
    """Wrap raw PCM in a WAV header so we can feed it back to Gemini."""
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate)
        w.writeframes(pcm)
    return buf.getvalue()

def play(pcm, rate=24000):
    """Play PCM audio right here in the notebook."""
    return Audio(np.frombuffer(pcm, dtype=np.int16), rate=rate)

print("✅ Client ready")

✅ Client ready


## 1. The AI Speaks

Give the TTS model text; it returns **raw PCM samples** (not an MP3 file). We play them straight in the notebook. Press play on the bar below — that's Gemini's voice.

In [16]:
pcm = tts("Hey! I'm your AI study buddy. Ask me anything.")
play(pcm)

## What's PCM (and why WAV)?

Gemini's TTS doesn't return an `.mp3` or `.wav` file — it returns **PCM**: the raw sound wave as a long list of numbers (here, 24,000 samples per second, each a 16-bit value, one mono channel). That *is* the audio, but there's no header telling a player "this is audio, at this sample rate" — so most players can't open it directly. (Above, we played it by handing the samples **+ the rate** straight to the notebook.)

A **WAV file** is just that same PCM data with a tiny header bolted onto the front describing the format. So `to_wav()` doesn't re-encode anything — it *wraps* the raw samples so the bytes become a real file that Gemini (and any audio player) recognizes. That's exactly what we need in the next step to send audio back *in*.

## 2. The AI Hears

Now the other direction: feed **audio in** and the model understands the spoken words.

To keep this notebook self-contained we *synthesize* the question with TTS — in a real app this is a recording of **your** voice from a mic. We wrap the PCM in a WAV header, hand it to `gemini-2.5-flash`, and ask it to transcribe.

In [17]:
question_pcm = tts("What is the time complexity of binary search?")

heard = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=["Transcribe exactly what is said in this audio.",
              types.Part.from_bytes(data=to_wav(question_pcm), mime_type="audio/wav")])

print("🎧 The AI heard:", heard.text.strip())
play(question_pcm)

🎧 The AI heard: What is the time complexity of binary search?


## 3. The Full Voice Loop

Chain both directions into one function: **speak → hear → think → speak back**. The single `gemini-2.5-flash` call both *listens* to the audio and *answers* it — then we voice the answer.

In [18]:
def voice_buddy(question_text, voice="Kore"):
    # 1) YOU talk (synthesized here for a self-contained demo; in real use, your mic).
    you_pcm = tts(question_text)

    # 2) The AI HEARS the audio and ANSWERS it — one gemini-2.5-flash call.
    resp = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=["You are a concise study buddy. Listen to the spoken question and answer in ONE short sentence.",
                  types.Part.from_bytes(data=to_wav(you_pcm), mime_type="audio/wav")])
    answer = resp.text.strip()

    # 3) The AI SPEAKS the answer back.
    return answer, tts(answer)

answer, answer_pcm = voice_buddy("What's the time complexity of binary search?")
print("💬 The AI's answer:", answer)
play(answer_pcm)

💬 The AI's answer: The time complexity of binary search is O of log n.


## Key Takeaways

1. **Two models, one loop** — `gemini-2.5-flash-preview-tts` is the voice (text→speech), `gemini-2.5-flash` is the ears + brain (audio in → answer). Chain them and the AI holds a spoken conversation.
2. **TTS returns raw PCM, not a file** — 24kHz 16-bit mono samples. Wrap them in a WAV header with `wave` before you save them or feed them back.
3. **All on the free tier, no extra libs** — pure `google-genai`. ⚠️ The TTS/audio models are *preview* (their own quota, and they can change).

---

Next: Day 49 — Coming soon